# Raw Dataset Processor

Pipeline monitoring untuk mengubah raw dataset `datasets/url_discovery/*.json` menjadi kandidat dataset dengan schema seperti `datasets/v1_shs_datasets.csv`. Notebook ini tidak menyimpan output secara otomatis.

## 1. Setup

In [1]:
from pathlib import Path
import csv
import sys

MAX_CSV_FIELD_SIZE = sys.maxsize
while True:
    try:
        csv.field_size_limit(MAX_CSV_FIELD_SIZE)
        break
    except OverflowError:
        MAX_CSV_FIELD_SIZE //= 10

import polars as pl
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.dataset_service import DatasetService
from services.source_blacklist_service import SourceBlacklistService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(250)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)

dataset_service = DatasetService()
RAW_FOLDER = config.DATASETS / "url_discovery"
RESEARCH_CONFIG_PATH = config.RESEARCH_CONFIG_PATH
REFERENCE_DATASET_PATH = config.DATASETS / "reference_datasets.csv"
SOURCE_URL_BLACKLIST_PATH = config.SOURCE_URL_BLACKLIST_PATH
SOURCE_BLACKLIST_RULES_PATH = config.SOURCE_BLACKLIST_RULES_PATH
USED_CANDIDATE_BLACKLIST_FOLDER = config.OUTPUTS / "datasets"
USED_CANDIDATE_BLACKLIST_PATTERN = "v*_candidate_labeling_dataset.csv"

source_url_blacklist = SourceBlacklistService.load_exact_blacklist(
    SOURCE_URL_BLACKLIST_PATH
)
used_candidate_source_urls = dataset_service.load_used_candidate_source_urls(
    USED_CANDIDATE_BLACKLIST_FOLDER,
    pattern=USED_CANDIDATE_BLACKLIST_PATTERN,
)
source_blacklist_service = SourceBlacklistService(
    exact_urls=[*source_url_blacklist, *used_candidate_source_urls],
    rules=SourceBlacklistService.load_rules(SOURCE_BLACKLIST_RULES_PATH),
)

print(f"Raw folder: {RAW_FOLDER}")
print(f"Research config: {RESEARCH_CONFIG_PATH}")
print(f"Reference schema: {REFERENCE_DATASET_PATH}")
print(f"Source URL blacklist: {SOURCE_URL_BLACKLIST_PATH}")
print(f"Source blacklist rules: {SOURCE_BLACKLIST_RULES_PATH}")
print(f"Used candidate blacklist folder: {USED_CANDIDATE_BLACKLIST_FOLDER}")
print(f"Used candidate source URLs: {len(used_candidate_source_urls):,}")


Raw folder: E:\School\tugas-akhir\project\datasets\url_discovery
Research config: E:\School\tugas-akhir\project\resources\research_config.json
Reference schema: E:\School\tugas-akhir\project\datasets\reference_datasets.csv
Source URL blacklist: E:\School\tugas-akhir\project\resources\source_url_blacklist.json
Source blacklist rules: E:\School\tugas-akhir\project\resources\source_blacklist_rules.json
Used candidate blacklist folder: E:\School\tugas-akhir\project\outputs\datasets
Used candidate source URLs: 325


## 2. Load Config dan Raw Discovery

In [2]:
research_config = dataset_service.load_research_config(RESEARCH_CONFIG_PATH)
meta_df = dataset_service.load_url_discovery_meta(RAW_FOLDER)
queries_df = dataset_service.load_url_discovery_queries(RAW_FOLDER)
raw_records_df = dataset_service.load_url_discovery_records(RAW_FOLDER)

RECORDS_FILTER_CONDITIONS = [
    # pl.col("content_status") == "success",
    # pl.col("content_text").fill_null("").str.strip_chars().str.len_chars() > 0,
    # ~pl.col("domain").str.contains("instagram.com|tiktok.com|facebook.com"),
]


def apply_filter_conditions(df: pl.DataFrame, conditions: list) -> pl.DataFrame:
    if not conditions:
        return df
    expression = conditions[0]
    for condition in conditions[1:]:
        expression = expression & condition
    return df.filter(expression)


records_df = apply_filter_conditions(raw_records_df, RECORDS_FILTER_CONDITIONS)

print("Search label:", research_config.get("search_label"))
print("Primary location:", research_config.get("primary_location"))
print(f"File metadata: {meta_df.height:,}")
print(f"Query rows: {queries_df.height:,}")
print(f"Raw records unik: {raw_records_df.height:,}")
print(f"Records setelah filter manual: {records_df.height:,}")

display(meta_df)

Search label: SHS/PLTS Kalimantan Barat
Primary location: Kalimantan Barat
File metadata: 9
Query rows: 2,537
Raw records unik: 1,521
Records setelah filter manual: 1,521


source_file,built_at,search_label,source_files,record_count,content_count,query_count
"""dataset_20260627T094855Z.json""","""2026-06-27T09:48:55.240893+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",214,214,335
"""dataset_20260627T101253Z.json""","""2026-06-27T10:12:53.267116+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",203,203,268
"""dataset_20260627T135259Z.json""","""2026-06-27T13:52:59.725144+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",88,88,197
"""dataset_20260628T033631Z.json""","""2026-06-28T03:36:31.219965+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",202,202,381
"""dataset_20260628T104512Z.json""","""2026-06-28T10:45:12.810124+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",136,136,369
"""dataset_20260628T113525Z.json""","""2026-06-28T11:35:25.172805+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",12,12,28
"""dataset_20260628T173013Z.json""","""2026-06-28T17:30:13.810625+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",250,250,267
"""dataset_20260630T005918Z.json""","""2026-06-30T00:59:18.792990+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",259,259,490
"""dataset_20260630T114825Z.json""","""2026-06-30T11:48:25.984980+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",157,156,470


## 3. Monitoring Kualitas Raw Dataset

In [3]:
status_distribution = (
    records_df.with_columns(pl.col("content_status").fill_null("missing"))
    .group_by("content_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

domain_distribution = (
    records_df.with_columns(pl.col("domain").fill_null("unknown"))
    .group_by("domain")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

query_group_distribution = (
    records_df.with_columns(pl.col("query_group").fill_null("unknown"))
    .group_by("query_group")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

text_length_summary = records_df.select(
    pl.len().alias("total_records"),
    (
        pl.col("content_text")
        .fill_null("")
        .str.strip_chars()
        .str.len_chars()
        > 0
    ).sum().alias("records_with_text"),
    pl.col("content_text_length").min().alias("min_text_length"),
    pl.col("content_text_length").mean().round(2).alias("avg_text_length"),
    pl.col("content_text_length").max().alias("max_text_length"),
)

display(status_distribution)
display(domain_distribution)
display(query_group_distribution)
display(text_length_summary)

content_status,jumlah
"""success""",1200
"""too_short""",229
"""pdf_skipped""",43
"""failed""",39
"""pending""",9
"""""",1


domain,jumlah
"""www.instagram.com""",596
"""www.tiktok.com""",223
"""www.facebook.com""",65
"""pontianakpost.jawapos.com""",36
"""rri.co.id""",29
"""kalbar.antaranews.com""",28
"""krisnamandiri.com""",12
"""www.researchgate.net""",12
"""id.scribd.com""",11
"""gatrik.esdm.go.id""",9


query_group,jumlah
"""core:electricity_access""",232
"""issue:access""",163
"""core:grid_hybrid""",115
"""core:household_solar""",103
"""issue:benefit""",93
"""issue:problem""",84
"""social""",75
"""core:technical""",71
"""core:village_solar""",68
"""core:government_program""",67


total_records,records_with_text,min_text_length,avg_text_length,max_text_length
1521,1236,0,3743.0,713441


## 4. Bangun Kandidat Schema v1

In [4]:
candidate_df = dataset_service.build_v1_candidate_rows(
    records_df=records_df,
    research_config=research_config,
)
candidate_df = dataset_service.enrich_source_blacklist_status(
    candidate_df,
    source_blacklist_service,
)

CANDIDATE_FILTER_CONDITIONS = [
    # pl.col("dataset_tier") == "C_holdout_excluded",
    # pl.col("source_type") == "social_media",
    # pl.col("source_url").str.to_lowercase().str.contains("facebook.com"),
    # ~pl.col("text").str.to_lowercase().str.contains("penayangan"),
    # pl.col("aspect").is_in(["experience", "benefit", "access"]),
]

filtered_candidate_df = apply_filter_conditions(
    candidate_df, CANDIDATE_FILTER_CONDITIONS
)
blacklist_audit_df = dataset_service.filter_blacklist_audit_candidates(
    filtered_candidate_df
)
clear_candidate_df = dataset_service.filter_clear_source_candidates(
    filtered_candidate_df
)

reference_df = dataset_service.load(REFERENCE_DATASET_PATH)
expected_columns = list(dataset_service.v1_dataset_columns())
missing_candidate_columns = [
    column for column in expected_columns if column not in clear_candidate_df.columns
]
extra_candidate_columns = [
    column for column in clear_candidate_df.columns if column not in expected_columns
]


## 5. Dataset Kandidat Siap Labeling


In [5]:
MAX_ROWS_PER_VALUE = 10
LABELING_DATASET_TIERS = ["A_candidate_core", "B_review_queue"]

labeling_ready_df = dataset_service.filter_labeling_ready_candidates(
    clear_candidate_df
)
labeling_df = dataset_service.build_candidate_labeling_dataset(
    labeling_ready_df,
    max_rows_per_value=MAX_ROWS_PER_VALUE,
)

candidate_combination_df = dataset_service.summarize_candidate_combinations(
    labeling_ready_df
).rename({"jumlah": "jumlah_tersedia"})

valid_location_df = labeling_ready_df.filter(
    (pl.col("is_specific_location") == True)
    & pl.col("location_source").is_in(["text", "query"])
)
valid_location_combination_df = dataset_service.summarize_candidate_combinations(
    valid_location_df
).rename({"jumlah": "jumlah_lokasi_valid"})

labeling_combination_df = dataset_service.summarize_labeling_selection_buckets(
    labeling_df
).rename({"jumlah": "jumlah_diambil"})

combination_audit_df = (
    candidate_combination_df
    .join(
        valid_location_combination_df,
        on=["combination_column", "combination_value"],
        how="left",
    )
    .join(
        labeling_combination_df,
        on=["combination_column", "combination_value"],
        how="left",
    )
    .with_columns(
        pl.col("jumlah_lokasi_valid").fill_null(0).cast(pl.Int64),
        pl.col("jumlah_diambil").fill_null(0).cast(pl.Int64),
    )
)

blacklist_status_distribution_df = (
    filtered_candidate_df.group_by("blacklist_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print(f"Candidate rows dikeluarkan blacklist: {blacklist_audit_df.height:,}")
print(f"Candidate rows clear: {clear_candidate_df.height:,}")
print(f"Candidate rows tier A/B siap labeling: {labeling_ready_df.height:,}")
print(f"Candidate rows dengan lokasi valid: {valid_location_df.height:,}")
print(f"Dataset tier labeling: {LABELING_DATASET_TIERS}")
print(f"Final siap labeling: {labeling_df.height:,}")
print(f"Maksimal data per nilai unik: {MAX_ROWS_PER_VALUE}")

display(blacklist_status_distribution_df)
display(combination_audit_df)


Candidate rows setelah filter kandidat: 1,521
Candidate rows dikeluarkan blacklist: 441
Candidate rows clear: 1,080
Candidate rows tier A/B siap labeling: 577
Candidate rows dengan lokasi valid: 577
Dataset tier labeling: ['A_candidate_core', 'B_review_queue']
Final siap labeling: 70
Maksimal data per nilai unik: 10


blacklist_status,jumlah
"""clear""",1080
"""original_exact_blacklist""",374
"""discovered_pattern_blacklist""",67


combination_column,combination_value,jumlah_tersedia,jumlah_lokasi_valid,jumlah_diambil
"""source_type""","""online_news""",206,206,10
"""source_type""","""social_media""",371,371,10
"""aspect""","""benefit""",38,38,10
"""aspect""","""electricity_access""",18,18,10
"""aspect""","""environment""",18,18,10
"""aspect""","""experience""",389,389,10
"""aspect""","""general_shs""",114,114,10


## 6. Preview Lokasi Valid


In [6]:
valid_location_columns = [
    "text_id",
    "location",
    "location_source",
    "location_match",
    "blacklist_status",
    "source_type",
    "aspect",
    "dataset_tier",
    "source_url",
    "query",
    "text",
]
valid_location_columns = [
    column for column in valid_location_columns if column in valid_location_df.columns
]

display(valid_location_df.select(valid_location_columns).head(10))


text_id,location,location_source,location_match,blacklist_status,source_type,aspect,dataset_tier,source_url,query,text
"""RAW-0003""","""Sambas""","""query""","""sambas""","""clear""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/reel/DY0bixATBPD""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""OIS POWER on Instagram: ""#PLTS #PanelSurya #SolarPanel #EnergiTerbarukan #SolarEnergy"" OIS POWER | #PLTS #PanelSurya #SolarPanel ... Instagram · ois_power 6 suka · 1 bulan yang lalu 6 likes, 0 comments - ois_power on May 26, 2026: ""#PLTS #PanelSurya …"
"""RAW-0004""","""Sambas""","""query""","""sambas""","""clear""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/p/DRq794LgAB2""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""TEKNIK LISTRIK on Instagram: ""Panel Surya Listrik (Fotovoltaik/PV) adalah teknologi yang mengambil energi dari cahaya matahari (foton), bukan panas, lalu mengubahnya menjadi listrik DC yang kemudian diubah oleh Inverter menjadi listrik AC yang dapat …"
"""RAW-0005""","""Sambas""","""query""","""sambas""","""clear""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/reel/DQlfNxOATsq""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""Datascrip Service Center on Instagram: ""Waktu yg tepat untuk membersihkan panel surya."" Waktu yg tepat untuk membersihkan panel surya. Instagram · datascrip.service.center.id 1 suka · 7 bulan yang lalu 1 likes, 0 comments - datascrip.service.center.i…"
"""RAW-0007""","""Sambas""","""query""","""sambas""","""clear""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/p/DP2n1gQD6J_""","""""tenaga surya"" ""listrik 24 jam"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""Damai Cable Indonesia on Instagram: ""Butuh kabel panel surya? Kami siap penuhi kebutuhan proyek energi terbarukanmu! Kuat, awet, dan sudah teruji kualitasnya. #damaicable"" Butuh kabel panel surya? Kami siap penuhi kebutuhan ... Instagram · damaicable…"
"""RAW-0009""","""Singkawang""","""text""","""singkawang""","""clear""","""online_news""","""experience""","""A_candidate_core""","""https://www.suarakalbar.co.id/2025/01/wujud-negara-hadir-pemerintah-dan-pln-berhasil-listriki-9992-persen-desa-di-seluruh-indonesia""","""""tenaga surya"" ""listrik 24 jam"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""Wujud Negara Hadir, Pemerintah dan PLN Berhasil Listriki 99,92 Persen Desa di Seluruh Indonesia Wujud Negara Hadir, Pemerintah dan PLN Berhasil Listriki ... suarakalbar.co.id https://www.suarakalbar.co.id › Nasional Jakarta (Suara Kalbar)- Pemerintah…"
"""RAW-0010""","""Sungai Raya Kepulauan""","""text""","""sungai raya kepulauan""","""clear""","""online_news""","""experience""","""A_candidate_core""","""https://www.liputanpontianak.com/warga-pulau-lemukutan-harapkan-listrik-24-jam-untuk-dongkrak-pariwisata""","""""energi terbarukan"" ""warga"" ""Pulau Lemukutan"" after:2024-01-01 before:2026-06-27""","""Warga Pulau Lemukutan Harapkan Listrik 24 Jam untuk Dongkrak Pariwisata Warga Pulau Lemukutan Harapkan Listrik 24 Jam untuk ... Liputan Pontianak https://www.liputanpontianak.com › warga-pulau-lemu... Warga Pulau Lemukutan Bengkayang berharap listrik…"
"""RAW-0011""","""Bengkayang""","""text""","""pulau lemukutan""","""clear""","""social_media""","""experience""","""B_review_queue""","""https://www.facebook.com/liputanpontianak/videos/warga-pulau-lemukutan-bengkayang-masih-mendambakan-listrik-24-jam-penuh-saat-ini/1155390963120098""","""""energi terbarukan"" ""warga"" ""Pulau Lemukutan"" after:2024-01-01 before:2026-06-27""","""4.5K views · 17 reactions | Warga Pulau Lemukutan, Bengkayang, masih mendambakan listrik 24 jam penuh. Saat ini, listrik hanya menyala 14 jam sehari, padahal Pulau Lemukutan sudah ditetapkan sebagai Desa Wisata 

## 7. Final Dataset Siap Labeling


In [7]:
target_columns = list(dataset_service.v1_dataset_columns())
monitoring_columns = [
    "location_source",
    "location_match",
    "is_specific_location",
    "blacklist_status",
    "blacklist_reason_codes",
    "normalized_source_url",
    "blacklist_is_excluded",
    "raw_source_file",
    "raw_domain",
    "content_status",
    "query_group",
    "query",
    "raw_title",
    "raw_text_length",
]
final_labeling_columns = [
    column for column in [*target_columns, *monitoring_columns]
    if column in labeling_df.columns
]

missing_final_columns = [
    column for column in target_columns if column not in labeling_df.columns
]
extra_final_columns = [
    column for column in labeling_df.columns if column not in target_columns
]

print("Kolom target hilang:", missing_final_columns)
print("Kolom monitoring tambahan:", extra_final_columns)
print(
    "Urutan kolom target sama:",
    labeling_df.columns[: len(target_columns)] == target_columns,
)

display(labeling_df.select(final_labeling_columns))


Kolom target hilang: []
Kolom monitoring tambahan: ['location_source', 'location_match', 'is_specific_location', 'labeling_bucket_column', 'labeling_bucket_value', 'raw_source_file', 'raw_domain', 'content_status', 'raw_title', 'raw_text_length', 'query_group', 'query', 'blacklist_status', 'blacklist_reason_codes', 'normalized_source_url', 'blacklist_is_excluded']
Urutan kolom target sama: False


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note,location_source,location_match,is_specific_location,blacklist_status,blacklist_reason_codes,normalized_source_url,blacklist_is_excluded,raw_source_file,raw_domain,content_status,query_group,query,raw_title,raw_text_length
"""RAW-1000""","""PT Sambas informasi Entertainment on Instagram: ""Info perbaikan & pemadaman listrik (Assalamualaikum.wr.b Izin bapak ibu untuk daerah kota sambas & sekitarnya akan dilakukan padam emergency untuk perbaikan peralatan jaringan listrik keluar api didaer…","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""electricity_access""","""Sambas""","""""","""unlabeled""","""RAW-SRC-1000""","""social_media""","""https://www.instagram.com/reel/C5BY3q1vmcl""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review""","""text""","""sambas""",true,"""clear""","""[]""","""https://instagram.com/reel/C5BY3q1vmcl""",false,"""dataset_20260628T173013Z.json""","""www.instagram.com""","""success""","""core:electricity_access""","""""pemadaman listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""Info perbaikan & pemadaman listrik (Assalamualaikum.wr.b ...""",667
"""RAW-1066""","""plnup3pontianak on Instagram: ""Stimulus Pemerintah melalui Diskon Tarif Listrik 50% untuk pelanggan rumah tangga daya 450 VA hingga 2.200 VA sudah berlaku hingga akhir Februari 2025 loh !! #PLN #PLNUntukIndonesia #AcceleratingRenewableEnergy #Swasemb…","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""electricity_access""","""Pontianak""","""""","""unlabeled""","""RAW-SRC-1066""","""social_media""","""https://www.instagram.com/p/DEgo-1_yd0F""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review""","""query""","""pontianak""",true,"""clear""","""[]""","""https://instagram.com/p/DEgo-1_yd0F""",false,"""dataset_20260628T173013Z.json""","""www.instagram.com""","""success""","""core:electricity_access""","""""tarif listrik"" ""Pontianak"" after:2024-01-01 before:2026-06-27""","""Stimulus Pemerintah melalui Diskon Tarif Listrik 50 ...""",653
"""RAW-0868""","""Pontianak Post on Instagram: ""DPR RI Tinjau Pembangunan Listrik Desa PLN di Kubu Raya, Perkuat Pemerataan Akses Energi hingga Pelosok Baca selengkapnya: https://pontianakpost.jawapos.com/pln/2605110032/dpr-ri-tinjau-pembangunan-listrik-desa-pln-di-ku…","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""electricity_access""","""Kubu Raya""","""""","""unlabeled""","""RAW-SRC-0868""","""social_media""","""https://www.instagram.com/p/DYMPSzRmdnM""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review""","""text""","""kubu raya""",true,"""clear""","""[]""","""https://instagram.com/p/DYMPSzRmdnM""",false,"""dataset_20260628T173013Z.json""","""www.instagram.com""","""success""","""core:electricity_access""","""""listrik desa"" ""Kayong Utara"" after:2024-01-01 before:2026-06-27""","""DPR RI Tinjau Pembangunan Listrik Desa PLN di Kubu ...""",650
"""RAW-0692""","""Landak Pusat Informasi on Instagram: ""Informasi jika ada pemadaman listrik 3 November 2024 #landakpusatinformasi"" Informasi jika ada pemadaman listrik 3 November 2024 ... Instagram · landakpusatinformasi 1,3 rb+ suka · 1 tahun yang lalu 1,276 likes, …","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""electricity_access""","""Landak""","""""","""unlabeled""","""RAW-SRC-0692""","""social_media""","""https://www.instagram.com/reel/DB52Z7FSfZy""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""soci

In [8]:
tier_distribution = (
    clear_candidate_df.group_by("dataset_tier")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

inclusion_distribution = (
    clear_candidate_df.group_by("inclusion_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

source_type_distribution = (
    clear_candidate_df.group_by("source_type")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

aspect_distribution = (
    clear_candidate_df.group_by("aspect")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)


location_distribution = (
    clear_candidate_df.group_by("location")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

display(tier_distribution)
display(inclusion_distribution)
display(source_type_distribution)
display(aspect_distribution)
display(location_distribution)


dataset_tier,jumlah
"""B_review_queue""",489
"""A_candidate_core""",308
"""C_holdout_excluded""",283


inclusion_status,jumlah
"""review_required_before_core""",489
"""candidate_analysis_ready""",308
"""held_out_not_for_sentiment_core""",283


source_type,jumlah
"""social_media""",713
"""online_news""",309
"""academic_repository""",39
"""government_official""",19


aspect,jumlah
"""experience""",529
"""general_shs""",355
"""benefit""",74
"""electricity_access""",50
"""environment""",37
"""grid_hybrid""",25
"""household_solar""",5
"""problem""",3
"""village_solar""",2


location,jumlah
"""Kalimantan Barat""",231
"""Pontianak""",70
"""Sambas""",64
"""Ketapang""",63
"""Sintang""",60
"""Kubu Raya""",59
"""Kapuas Hulu""",58
"""Singkawang""",48
"""Landak""",47
"""""",47


In [9]:
print(f"Candidate rows sebelum filter kandidat: {candidate_df.height:,}")
print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print(f"Candidate rows clear setelah blacklist: {clear_candidate_df.height:,}")
print("Kolom wajib hilang:", missing_candidate_columns)
print("Kolom monitoring tambahan:", extra_candidate_columns)
print(
    "Urutan kolom utama sama:",
    clear_candidate_df.columns[: len(expected_columns)] == expected_columns,
)

display(clear_candidate_df.select(expected_columns).head(20))


Candidate rows sebelum filter kandidat: 1,521
Candidate rows setelah filter kandidat: 1,521
Candidate rows clear setelah blacklist: 1,080
Kolom wajib hilang: []
Kolom monitoring tambahan: ['raw_source_file', 'raw_domain', 'content_status', 'query_group', 'query', 'raw_title', 'raw_text_length', 'location_source', 'location_match', 'is_specific_location', 'blacklist_status', 'blacklist_reason_codes', 'normalized_source_url', 'blacklist_is_excluded']
Urutan kolom utama sama: True


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note
"""RAW-0003""","""OIS POWER on Instagram: ""#PLTS #PanelSurya #SolarPanel #EnergiTerbarukan #SolarEnergy"" OIS POWER | #PLTS #PanelSurya #SolarPanel ... Instagram · ois_power 6 suka · 1 bulan yang lalu 6 likes, 0 comments - ois_power on May 26, 2026: ""#PLTS #PanelSurya …","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0003""","""social_media""","""https://www.instagram.com/reel/DY0bixATBPD""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0004""","""TEKNIK LISTRIK on Instagram: ""Panel Surya Listrik (Fotovoltaik/PV) adalah teknologi yang mengambil energi dari cahaya matahari (foton), bukan panas, lalu mengubahnya menjadi listrik DC yang kemudian diubah oleh Inverter menjadi listrik AC yang dapat …","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0004""","""social_media""","""https://www.instagram.com/p/DRq794LgAB2""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0005""","""Datascrip Service Center on Instagram: ""Waktu yg tepat untuk membersihkan panel surya."" Waktu yg tepat untuk membersihkan panel surya. Instagram · datascrip.service.center.id 1 suka · 7 bulan yang lalu 1 likes, 0 comments - datascrip.service.center.i…","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0005""","""social_media""","""https://www.instagram.com/reel/DQlfNxOATsq""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0006""","""TikTok - Make Your Day Listrik Padam Sumatera: Solusi PLTS dan Stok Cadangan ... TikTok · evanstoryy 7,2 rb+ penayangan @evanstoryyy 7643019382251080967""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0006""","""social_media""","""https://www.tiktok.com/@evanstoryyy/video/7643019382251080967""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review"""
"""RAW-0007""","""Damai Cable Indonesia on Instagram: ""Butuh kabel panel surya? Kami siap penuhi kebutuhan proyek energi terbarukanmu! Kuat, awet, dan sudah teruji kualitasnya. #damaicable"" Butuh kabel panel surya? Kami siap penuhi kebutuhan ... Instagram · damaicable…","""public_expectation""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0007""","""social_media""","""https://www.instagram.com/p/DP2n1gQD6J_""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0008""","""TikTok - Make Your Day Teknologi CSP China di Gurun Gobi: Pembangkit Listrik 24 ... TikTok · bacaaja.co 3,8 rb+ penayangan @bacaaja.co 7651132761448647956""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0008""","""social_media""","""https://www.tiktok.com/@bacaaja.co/video/7651132761448647956""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review"""
"""RAW-0009""","""Wujud Negara Hadir, Pemerintah dan PLN Berhasil Listriki 99,92 Persen Desa di Seluruh Indonesia Wujud Negara Hadir, Pemerinta